# 04 — Model Comparison Analysis

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("../.." ).resolve()))
sys.path.insert(0, str(Path(".").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.evaluation import ResultsManager

RESULTS_DIR = Path("results")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

rm = ResultsManager(RESULTS_DIR)
df = rm.load_all_results()
print(f"Total experiments loaded: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
print("Experiments by mode:")
print(df["mode"].value_counts())
print("\nExperiments by model:")
print(df["model_name"].value_counts())
print("\nExperiments by ticker:")
print(df["ticker"].value_counts().to_string())

## 1. Overall Rankings — F1-macro

In [ ]:
rank = (
    df.groupby(["model_name", "mode"])["ml_f1_macro"]
    .agg(["mean", "std", "count"])
    .round(4)
    .sort_values("mean", ascending=False)
)
rank.columns = ["F1-macro mean", "F1-macro std", "n_experiments"]
print("=== Model × Mode — F1-macro (mean across all tickers) ===")
print(rank.to_string())

## 2. Wavelet Impact: raw vs db4 vs learned_wavelet

In [ ]:
dl_df = df[df["mode"].isin(["raw", "db4", "learned_wavelet"])].copy()
fig, axes = plt.subplots(1, 4, figsize=(18, 5), sharey=True)
models = ["CNN", "LSTM", "CNN_LSTM", "Transformer"]
for ax, model in zip(axes, models):
    sub = dl_df[dl_df["model_name"] == model]
    if sub.empty:
        continue
    order = ["raw", "db4", "learned_wavelet"]
    sns.boxplot(data=sub, x="mode", y="ml_f1_macro", order=order, ax=ax,
                palette=["#95a5a6", "#3498db", "#e74c3c"])
    ax.set_title(model, fontsize=11)
    ax.set_xlabel("")
    ax.set_ylabel("F1-macro" if ax == axes[0] else "")
    ax.tick_params(axis="x", rotation=30)
plt.suptitle("F1-macro by Mode — raw vs db4 vs Learned Wavelet", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Best Model per Ticker

In [ ]:
best_per_ticker = (
    df.loc[df.groupby("ticker")["ml_f1_macro"].idxmax()]
    [["ticker", "model_name", "mode", "ml_f1_macro", "fin_sharpe"]]
    .set_index("ticker")
    .sort_values("ml_f1_macro", ascending=False)
)
print("Best model per ticker:")
print(best_per_ticker.round(4).to_string())
print(f"\nMost common best model: {best_per_ticker['model_name'].mode().iloc[0]}")
print(f"Most common best mode:  {best_per_ticker['mode'].mode().iloc[0]}")

## 4. Heatmap — F1-macro by Ticker × Model

In [ ]:
pivot = df.pivot_table(
    index="ticker", columns="model_name", values="ml_f1_macro", aggfunc="max"
).round(3)

fig, ax = plt.subplots(figsize=(14, 9))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGn", vmin=0.25, vmax=0.65,
            linewidths=0.3, ax=ax, annot_kws={"size": 8})
ax.set_title("F1-macro — Best result per Ticker × Model", fontsize=12)
plt.tight_layout()
plt.show()

## 5. ML vs DL Comparison

In [ ]:
df["category"] = df["mode"].apply(
    lambda m: "ML" if m == "ml" else ("DL-raw" if m == "raw" else ("DL-db4" if m == "db4" else "DL-learned"))
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric in zip(axes, ["ml_f1_macro", "fin_sharpe"]):
    if metric not in df.columns:
        continue
    order = ["ML", "DL-raw", "DL-db4", "DL-learned"]
    palette = {"ML": "#2ecc71", "DL-raw": "#95a5a6", "DL-db4": "#3498db", "DL-learned": "#e74c3c"}
    sns.boxplot(data=df, x="category", y=metric, order=order, palette=palette, ax=ax)
    ax.set_title(metric.replace("ml_", "").replace("fin_", "").replace("_", " ").title())
    ax.set_xlabel("")
plt.suptitle("ML vs DL Approaches", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Statistical Test — Learned Wavelet vs Baselines

In [ ]:
dl = df[df["mode"].isin(["raw", "db4", "learned_wavelet"])].copy()
lw_f1  = dl[dl["mode"] == "learned_wavelet"]["ml_f1_macro"].dropna()
raw_f1 = dl[dl["mode"] == "raw"]["ml_f1_macro"].dropna()
db4_f1 = dl[dl["mode"] == "db4"]["ml_f1_macro"].dropna()

for name, other in [("raw", raw_f1), ("db4", db4_f1)]:
    t_stat, p_val = stats.ttest_ind(lw_f1, other, alternative="greater")
    print(f"LW vs {name}: t={t_stat:.3f}  p={p_val:.4f}  "
          f"(LW mean={lw_f1.mean():.4f}  {name} mean={other.mean():.4f})")

## 7. Stability — F1 std across Folds

In [ ]:
std_col = "ml_f1_macro_std"
if std_col in df.columns:
    stability = (
        df.groupby(["model_name", "mode"])[std_col]
        .mean().round(4)
        .sort_values()
    )
    print("F1-macro std (lower = more stable):")
    print(stability.to_string())
else:
    print(f"Column '{std_col}' not found — run experiments first.")